# 🌿 XGBoost Fertilizer Dosage Predictor
**Dataset:** `Agri3D_LeafyGreens_Dataset_CLEANED.csv`  
**Goal:** Predict the **optimal total fertilizer dosage (kg/ha)** from soil & environmental features using XGBoost Regressor.

---
### 📋 Notebook Outline
1. Environment Setup
2. Data Loading & Exploration
3. Target Variable Engineering (kg/ha → optional ml conversion)
4. Preprocessing (Encoding + Scaling)
5. Feature Selection
6. Model Training (XGBoost Regressor)
7. Evaluation (RMSE, R²)
8. Visualization (Feature Importance + Actual vs Predicted)
9. Prediction Function

---
## 1. 📦 Environment Setup

In [ ]:
# Install required packages (run once)
import subprocess, sys

packages = ['pandas', 'scikit-learn', 'xgboost', 'matplotlib', 'seaborn', 'numpy']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('✅ All packages installed successfully!')

In [ ]:
# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from xgboost import XGBRegressor

# Plotting style
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='darkgrid', palette='muted')

print('✅ Imports successful!')
import xgboost; print(f'   XGBoost version : {xgboost.__version__}')
import sklearn;  print(f'   Scikit-learn     : {sklearn.__version__}')
print(f'   Pandas           : {pd.__version__}')

---
## 2. 📂 Data Loading & Exploration

In [ ]:
# ── Load dataset ────────────────────────────────────────────────────────────
CSV_PATH = 'Agri3D_LeafyGreens_Dataset_CLEANED.csv'
df = pd.read_csv(CSV_PATH)

print(f'Dataset shape : {df.shape}  ({df.shape[0]:,} rows × {df.shape[1]} columns)')
print(f'\nColumn list:')
for i, c in enumerate(df.columns, 1):
    print(f'  {i:2d}. {c}  [{df[c].dtype}]')

In [ ]:
# Quick peek at the data
df.head(5)

In [ ]:
# Statistical summary of numeric columns
df.describe().round(3)

In [ ]:
# Missing value audit
null_counts = df.isnull().sum()
print('Missing values per column:')
print(null_counts[null_counts > 0] if null_counts.any() else '  🎉 No missing values found!')

In [ ]:
# Unique crops in dataset
print(f'Unique crops ({df["crop"].nunique()}):',
      sorted(df['crop'].unique().tolist()))
print(f'\nUnique fertilizers ({df["Recommended Fertilizer(s)"].nunique()}):')
for f in df['Recommended Fertilizer(s)'].unique()[:10]:
    print(f'  • {f}')

---
## 3. 🎯 Target Variable Engineering

The dataset has **three individual dosage columns** in kg/ha:
- `N_dosage_kg_per_ha`  – Nitrogen dosage
- `P2O5_dosage_kg_per_ha` – Phosphorus dosage  
- `K2O_dosage_kg_per_ha`  – Potassium dosage

We construct a **`total_dosage_kg_per_ha`** target as the sum of all three, which represents the total fertilizer application rate.

### 💧 ml Conversion Guide
Fertilizer dosages in agriculture are typically given as **weight-based (kg/ha)** measurements.  
To convert to **volume (ml/ha or L/ha)** you need the **specific gravity (density)** of the particular liquid fertilizer product:

```
volume (L/ha) = mass (kg/ha) / density (kg/L)
volume (ml/ha) = volume (L/ha) × 1000
```

**Typical liquid fertilizer densities:**
| Fertilizer Type | Density (kg/L) |
|---|---|
| UAN (32% N solution) | ~1.32 |
| Liquid NPK 9-18-9 | ~1.30 |
| MAP solution | ~1.40 |
| Generic liquid fertilizer | ~1.20–1.35 |

A **`LIQUID_DENSITY_KG_PER_L`** variable below lets you toggle the conversion.

In [ ]:
# ── Build the target column ─────────────────────────────────────────────────
df['total_dosage_kg_per_ha'] = (
    df['N_dosage_kg_per_ha'].fillna(0)
    + df['P2O5_dosage_kg_per_ha'].fillna(0)
    + df['K2O_dosage_kg_per_ha'].fillna(0)
)

# ── Optional: ml conversion ─────────────────────────────────────────────────
# Set USE_ML_TARGET = True  to predict in ml/ha instead of kg/ha
# Adjust LIQUID_DENSITY_KG_PER_L to match your specific fertilizer product.
USE_ML_TARGET           = False   # ← change to True to switch to ml/ha
LIQUID_DENSITY_KG_PER_L = 1.30   # kg per litre (adjust for your product)

if USE_ML_TARGET:
    df['total_dosage_ml_per_ha'] = (df['total_dosage_kg_per_ha'] / LIQUID_DENSITY_KG_PER_L) * 1000
    TARGET_COL   = 'total_dosage_ml_per_ha'
    TARGET_LABEL = 'Total Dosage (ml/ha)'
else:
    TARGET_COL   = 'total_dosage_kg_per_ha'
    TARGET_LABEL = 'Total Dosage (kg/ha)'

print(f'🎯 Target column : {TARGET_COL}')
print(f'   Unit          : {TARGET_LABEL}')
print(f'\n   Min   : {df[TARGET_COL].min():.2f}')
print(f'   Mean  : {df[TARGET_COL].mean():.2f}')
print(f'   Median: {df[TARGET_COL].median():.2f}')
print(f'   Max   : {df[TARGET_COL].max():.2f}')

In [ ]:
# Visualise the target distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(df[TARGET_COL], bins=60, kde=True, color='steelblue', ax=axes[0])
axes[0].set_title(f'Distribution of {TARGET_LABEL}', fontsize=13, fontweight='bold')
axes[0].set_xlabel(TARGET_LABEL)
axes[0].set_ylabel('Count')

sns.boxplot(y=df[TARGET_COL], color='lightcoral', ax=axes[1])
axes[1].set_title(f'Box Plot – {TARGET_LABEL}', fontsize=13, fontweight='bold')
axes[1].set_ylabel(TARGET_LABEL)

plt.tight_layout()
plt.savefig('target_distribution.png', bbox_inches='tight')
plt.show()
print('📊 Figure saved as target_distribution.png')

---
## 4. 🔧 Preprocessing — Encoding Categorical Variables

In [ ]:
# ── Define features ─────────────────────────────────────────────────────────
# Numeric soil / environmental features
NUMERIC_FEATURES = ['N', 'P', 'K', 'pH', 'S', 'Zn', 'Fe', 'Cu', 'Mn', 'B', 'OC',
                    'temperature', 'humidity', 'rainfall']

# Categorical: 'crop' → Label Encoded  (high-cardinality, ordinal-neutral)
# Categorical: 'Recommended Fertilizer(s)' → Label Encoded (too many levels for OHE)
# 'Dosage_Notes' → Label Encoded (represents application method category)
CATEGORICAL_LE  = ['crop', 'Recommended Fertilizer(s)', 'Dosage_Notes']

# Working copy
data = df[NUMERIC_FEATURES + CATEGORICAL_LE + [TARGET_COL]].copy()

print('Features used:')
print('  Numeric    :', NUMERIC_FEATURES)
print('  Categorical:', CATEGORICAL_LE)
print(f'\n  Total features : {len(NUMERIC_FEATURES) + len(CATEGORICAL_LE)}')
print(f'  Target         : {TARGET_COL}')

In [ ]:
# ── Label Encoding ──────────────────────────────────────────────────────────
le_encoders = {}  # store encoders for later use in the prediction function

for col in CATEGORICAL_LE:
    le = LabelEncoder()
    data[col] = data[col].fillna('Unknown')          # handle any NaN
    data[col + '_enc'] = le.fit_transform(data[col])
    le_encoders[col] = le
    print(f'  {col:35s} → {le.classes_.shape[0]} unique classes encoded')

# Final feature list after encoding (encoded columns replace originals)
ENC_CATS = [c + '_enc' for c in CATEGORICAL_LE]
FEATURE_COLS = NUMERIC_FEATURES + ENC_CATS

X = data[FEATURE_COLS]
y = data[TARGET_COL]

print(f'\n✅ Feature matrix X shape : {X.shape}')
print(f'   Target vector y shape  : {y.shape}')

In [ ]:
# ── Correlation heatmap (numeric features vs target) ────────────────────────
corr_data = data[NUMERIC_FEATURES + [TARGET_COL]].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_data, dtype=bool))
sns.heatmap(corr_data, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.4, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()
print('📊 Figure saved as correlation_heatmap.png')

---
## 5. ✂️ Train / Test Split

In [ ]:
# 80 / 20 stratified split (random_state for reproducibility)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print(f'Training set   : {X_train.shape[0]:,} samples  ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test set       : {X_test.shape[0]:,} samples  ({X_test.shape[0]/len(X)*100:.1f}%)')

---
## 6. 🚀 XGBoost Regressor Training

In [ ]:
# ── Hyperparameters ─────────────────────────────────────────────────────────
# These are well-balanced defaults; tune via GridSearchCV / Optuna for production.
xgb_params = dict(
    n_estimators      = 500,
    max_depth         = 7,
    learning_rate     = 0.05,
    subsample         = 0.80,
    colsample_bytree  = 0.80,
    min_child_weight  = 3,
    reg_alpha         = 0.1,    # L1 regularisation
    reg_lambda        = 1.0,    # L2 regularisation
    objective         = 'reg:squarederror',
    eval_metric       = 'rmse',
    random_state      = 42,
    n_jobs            = -1,
    verbosity         = 0
)

model = XGBRegressor(**xgb_params)

# ── Train with early stopping ────────────────────────────────────────────────
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    early_stopping_rounds=40,
    verbose=50
)

print(f'\n✅ Training complete!')
print(f'   Best iteration : {model.best_iteration}')

---
## 7. 📈 Model Evaluation

In [ ]:
# ── Predictions ─────────────────────────────────────────────────────────────
y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

# ── Metrics ─────────────────────────────────────────────────────────────────
def evaluate(y_true, y_pred, label=''):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f'  [{label}]')
    print(f'    RMSE : {rmse:>10.4f}  {TARGET_LABEL}')
    print(f'    MAE  : {mae:>10.4f}  {TARGET_LABEL}')
    print(f'    R²   : {r2:>10.4f}')
    return rmse, mae, r2

print('📊 Model Performance')
print('=' * 45)
train_rmse, train_mae, train_r2 = evaluate(y_train, y_pred_train, 'Train')
print()
test_rmse,  test_mae,  test_r2  = evaluate(y_test,  y_pred_test,  'Test ')
print('=' * 45)

# ── 5-fold Cross-Validation ─────────────────────────────────────────────────
print('\n📊 5-Fold Cross-Validation R² (on full dataset):')
cv_scores = cross_val_score(XGBRegressor(**xgb_params), X, y,
                             cv=5, scoring='r2', n_jobs=-1)
print(f'   Scores  : {np.round(cv_scores, 4)}')
print(f'   Mean R² : {cv_scores.mean():.4f}  ± {cv_scores.std():.4f}')

---
## 8. 📊 Visualization

In [ ]:
# ── 8a. Training Loss Curve ─────────────────────────────────────────────────
results = model.evals_result()
epochs  = len(results['validation_0']['rmse'])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(results['validation_0']['rmse'], label='Train RMSE', color='royalblue',   linewidth=1.8)
ax.plot(results['validation_1']['rmse'], label='Test RMSE',  color='tomato',      linewidth=1.8)
ax.axvline(model.best_iteration, color='gray', linestyle='--', alpha=0.7, label=f'Best iter ({model.best_iteration})')
ax.set_xlabel('Boosting Round')
ax.set_ylabel('RMSE')
ax.set_title('XGBoost Training & Validation RMSE', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('training_loss_curve.png', bbox_inches='tight')
plt.show()
print('📊 Figure saved as training_loss_curve.png')

In [ ]:
# ── 8b. Feature Importance (Gain) ──────────────────────────────────────────
importance_df = pd.DataFrame({
    'Feature'   : FEATURE_COLS,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(importance_df)))
bars = ax.barh(importance_df['Feature'], importance_df['Importance'], color=colors, edgecolor='white', linewidth=0.5)

# Annotate bars
for bar, val in zip(bars, importance_df['Importance']):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', ha='left', fontsize=8.5)

ax.set_xlabel('Feature Importance (Weight)', fontsize=11)
ax.set_title('XGBoost Feature Importance', fontsize=14, fontweight='bold')
ax.set_xlim(0, importance_df['Importance'].max() * 1.18)
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.show()
print('📊 Figure saved as feature_importance.png')

In [ ]:
# ── 8c. Actual vs Predicted scatter ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (y_true, y_pred, split_label, color) in zip(
        axes,
        [(y_train, y_pred_train, 'Train', 'royalblue'),
         (y_test,  y_pred_test,  'Test',  'tomato')]):

    ax.scatter(y_true, y_pred, alpha=0.35, s=15, color=color, edgecolors='none')

    # Perfect prediction line
    mn, mx = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
    ax.plot([mn, mx], [mn, mx], 'k--', linewidth=1.4, label='Perfect Fit')

    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    ax.set_title(f'{split_label} Set — Actual vs Predicted\nR²={r2:.4f}  RMSE={rmse:.2f}',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel(f'Actual {TARGET_LABEL}')
    ax.set_ylabel(f'Predicted {TARGET_LABEL}')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('actual_vs_predicted.png', bbox_inches='tight')
plt.show()
print('📊 Figure saved as actual_vs_predicted.png')

In [ ]:
# ── 8d. Residual Distribution ───────────────────────────────────────────────
residuals = y_test.values - y_pred_test

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(residuals, bins=60, kde=True, color='mediumpurple', ax=axes[0])
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_title('Residual Distribution (Test Set)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Residual (Actual − Predicted)')

axes[1].scatter(y_pred_test, residuals, alpha=0.3, s=12, color='mediumpurple', edgecolors='none')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title('Residuals vs Fitted (Test Set)', fontsize=12, fontweight='bold')
axes[1].set_xlabel(f'Predicted {TARGET_LABEL}')
axes[1].set_ylabel('Residual')

plt.tight_layout()
plt.savefig('residual_analysis.png', bbox_inches='tight')
plt.show()
print('📊 Figure saved as residual_analysis.png')

---
## 9. 🤖 Prediction Function

Use `predict_dosage()` to get a dosage estimate for any new soil/environmental reading.  
Only the **numeric soil & climate features** are required — crop and fertilizer type default to the most frequent values in the training data.

In [ ]:
# ── Build prediction function ────────────────────────────────────────────────

# Defaults: most frequent crop and fertilizer in dataset
_DEFAULT_CROP       = df['crop'].mode()[0]
_DEFAULT_FERTILIZER = df['Recommended Fertilizer(s)'].mode()[0]
_DEFAULT_DOSE_NOTE  = df['Dosage_Notes'].mode()[0]

def predict_dosage(
    N,
    P,
    K,
    pH,
    S            = 10.0,
    Zn           = 1.0,
    Fe           = 5.0,
    Cu           = 0.5,
    Mn           = 2.0,
    B            = 0.5,
    OC           = 1.5,
    temperature  = 25.0,
    humidity     = 70.0,
    rainfall     = 800.0,
    crop         = _DEFAULT_CROP,
    fertilizer   = _DEFAULT_FERTILIZER,
    dosage_notes = _DEFAULT_DOSE_NOTE,
):
    """
    Predict the total optimal fertilizer dosage.

    Required parameters
    -------------------
    N            : Soil Nitrogen  (kg/ha)
    P            : Soil Phosphorus (kg/ha)
    K            : Soil Potassium (kg/ha)
    pH           : Soil pH

    Optional parameters (with sensible defaults)
    --------------------------------------------
    S            : Soil Sulfur      (default 10  kg/ha)
    Zn           : Soil Zinc        (default 1   mg/kg)
    Fe           : Soil Iron        (default 5   mg/kg)
    Cu           : Soil Copper      (default 0.5 mg/kg)
    Mn           : Soil Manganese   (default 2   mg/kg)
    B            : Soil Boron       (default 0.5 mg/kg)
    OC           : Organic Carbon   (default 1.5 %)
    temperature  : Ambient temp     (default 25  °C)
    humidity     : Relative humidity(default 70  %)
    rainfall     : Annual rainfall  (default 800 mm)
    crop         : Crop type string (default most frequent in dataset)
    fertilizer   : Fertilizer name  (default most frequent in dataset)
    dosage_notes : Application mode (default most frequent in dataset)

    Returns
    -------
    float : Predicted dosage in the current TARGET_LABEL units.
    """
    # ── Encode categoricals (handle unseen labels gracefully) ────────────────
    def safe_encode(encoder, value, col_name):
        if value in encoder.classes_:
            return encoder.transform([value])[0]
        else:
            print(f'  ⚠️  Unseen {col_name} value "{value}". Falling back to default.')
            return encoder.transform([encoder.classes_[0]])[0]

    crop_enc       = safe_encode(le_encoders['crop'],                           crop,         'crop')
    fert_enc       = safe_encode(le_encoders['Recommended Fertilizer(s)'],      fertilizer,   'fertilizer')
    note_enc       = safe_encode(le_encoders['Dosage_Notes'],                   dosage_notes, 'dosage_notes')

    # ── Assemble feature row ─────────────────────────────────────────────────
    row = pd.DataFrame([{
        'N'           : N,
        'P'           : P,
        'K'           : K,
        'pH'          : pH,
        'S'           : S,
        'Zn'          : Zn,
        'Fe'          : Fe,
        'Cu'          : Cu,
        'Mn'          : Mn,
        'B'           : B,
        'OC'          : OC,
        'temperature' : temperature,
        'humidity'    : humidity,
        'rainfall'    : rainfall,
        'crop_enc'                       : crop_enc,
        'Recommended Fertilizer(s)_enc'  : fert_enc,
        'Dosage_Notes_enc'               : note_enc,
    }])[FEATURE_COLS]   # ensure column order matches training

    prediction = model.predict(row)[0]
    return float(prediction)


print('✅ predict_dosage() function is ready!')
print(f'   Default crop       : {_DEFAULT_CROP}')
print(f'   Default fertilizer : {_DEFAULT_FERTILIZER}')
print(f'   Default dosage note: {_DEFAULT_DOSE_NOTE}')

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 🌱 EXAMPLE PREDICTIONS  — Edit the values below for your soil readings!
# ──────────────────────────────────────────────────────────────────────────────

# Example 1: Typical moderate-nutrient soil
dosage_1 = predict_dosage(
    N=80,   P=40,   K=120,  pH=6.5,
    S=12,   Zn=0.8, Fe=4.5, Cu=0.4, Mn=1.8, B=0.4, OC=1.2,
    temperature=28, humidity=75, rainfall=900
)
print(f'Example 1 — Moderate soil    : {dosage_1:.2f} {TARGET_LABEL}')

# Example 2: Nutrient-poor soil (high dosage expected)
dosage_2 = predict_dosage(
    N=20,   P=10,   K=50,   pH=5.8,
    S=5,    Zn=0.3, Fe=2.0, Cu=0.2, Mn=0.8, B=0.2, OC=0.5,
    temperature=30, humidity=65, rainfall=600
)
print(f'Example 2 — Nutrient-poor   : {dosage_2:.2f} {TARGET_LABEL}')

# Example 3: Nutrient-rich soil (low dosage expected)
dosage_3 = predict_dosage(
    N=200,  P=120,  K=300,  pH=7.0,
    S=25,   Zn=2.5, Fe=15,  Cu=1.2, Mn=5,   B=1.0, OC=3.0,
    temperature=22, humidity=80, rainfall=1200
)
print(f'Example 3 — Nutrient-rich   : {dosage_3:.2f} {TARGET_LABEL}')

# Example 4: With specific crop and fertilizer
dosage_4 = predict_dosage(
    N=60, P=30, K=80, pH=6.2,
    temperature=26, humidity=72, rainfall=800,
    crop='kangkong',
    fertilizer='Urea (N); DAP (P); MOP (K)'
)
print(f'Example 4 — Kangkong crop   : {dosage_4:.2f} {TARGET_LABEL}')

In [ ]:
# ── ml Conversion Reference (single prediction) ─────────────────────────────
print('💧 Dosage Unit Conversion Reference')
print('=' * 45)
kg_ha_value = dosage_1
print(f'   kg/ha            : {kg_ha_value:.2f}')
for product, density in [('UAN (32% N)', 1.32), ('Liquid NPK 9-18-9', 1.30), ('Generic liquid', 1.25)]:
    ml_ha = (kg_ha_value / density) * 1000
    l_ha  =  kg_ha_value / density
    print(f'   {product:22s}: {l_ha:8.2f} L/ha  |  {ml_ha:10.2f} ml/ha  (density={density} kg/L)')

In [ ]:
# ── Save model ───────────────────────────────────────────────────────────────
model.save_model('xgboost_fertilizer_model.json')
print('✅ Model saved to : xgboost_fertilizer_model.json')
print('   Load it later with:')
print('     from xgboost import XGBRegressor')
print('     m = XGBRegressor()')
print('     m.load_model("xgboost_fertilizer_model.json")')

---
## ✅ Summary

| Item | Detail |
|---|---|
| **Dataset** | `Agri3D_LeafyGreens_Dataset_CLEANED.csv` — 9,600 rows × 25 cols |
| **Target** | `total_dosage_kg_per_ha` = N + P₂O₅ + K₂O dosages |
| **Features** | 14 numeric (soil + climate) + 3 label-encoded categoricals = 17 total |
| **Model** | XGBoost Regressor (n_estimators=500, early stopping) |
| **Split** | 80% train / 20% test |
| **Saved outputs** | `xgboost_fertilizer_model.json`, `feature_importance.png`, `actual_vs_predicted.png`, `training_loss_curve.png`, `residual_analysis.png`, `correlation_heatmap.png`, `target_distribution.png` |

### 📌 Next Steps
- **Hyperparameter tuning**: Use `GridSearchCV` or Bayesian optimisation (Optuna) to fine-tune `max_depth`, `learning_rate`, `n_estimators`.
- **SHAP values**: Run `shap.Explainer(model)(X_test)` for interpretable per-prediction feature attributions.
- **Separate N/P/K models**: Train three separate regressors if you need individual N, P₂O₅, K₂O predictions.
- **ml conversion**: Set `USE_ML_TARGET = True` (Cell 3) and adjust `LIQUID_DENSITY_KG_PER_L` if your workflow requires volume-based output.
